# SQL — Классификация и граф контрагентов

SQL-часть задания (Часть 1.3). База будет SQLite (`data/counterparty.db`).

**Три запроса:**
1. Топ-10 пар контрагентов по суммарному обороту
2. Месячный rolling-sum оборота по каждому отправителю (оконная функция)
3. Контрагенты, у которых >70% входящих платежей от одного источника

SQL-файлы лежат в `sql/`. Ноутбук их читает и выполняет — не дублирует код.

## 0. Подготовка: создаём базу

In [18]:
import sys
from pathlib import Path

import pandas as pd

# из notebooks/ поднимаемся в корень проекта, подключаем src/
ROOT = Path.cwd().parent
sys.path.append(str(ROOT / "src"))

from db import load_to_sqlite, run_query

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Создаём (или пересоздаём) базу из чистого CSV
load_to_sqlite(
    csv_path=ROOT / "data" / "transactions.csv",
    db_path=ROOT / "data" / "counterparty.db",
)

[db.py] Читаем и чистим C:\Users\emila\Test\counterparty-classification\data\transactions.csv ...
[db.py] Записываем 80,000 строк в C:\Users\emila\Test\counterparty-classification\data\counterparty.db ...
[db.py] Готово. База создана.


## 1. Топ-10 пар контрагентов по обороту

Запрос: `sql/top_pairs.sql`

In [20]:
sql = (ROOT / "sql" / "top_pairs.sql").read_text(encoding="utf-8")
df_pairs = run_query(sql, ROOT / "data" / "counterparty.db")
df_pairs

,sen,rec,cnt,sum
0,960819662090,711020691210,108,"300,712,858.72"
1,310405616795,551211641020,108,"259,874,422.42"
2,271023534832,850603506938,97,"256,545,776.71"
3,640522553440,730424495604,89,"236,862,505.78"
4,151008697404,740201661314,93,"229,751,033.77"
5,550624571494,850623490114,83,"229,578,734.61"
6,760616418322,410517502025,79,"212,504,488.97"
7,861121409930,121125400846,88,"204,639,115.68"
8,680213490446,900326509443,88,"202,055,181.69"
9,960406579569,221201693852,91,"186,844,150.64"


## 2. Месячный rolling-sum (нарастающий итог)

Запрос: `sql/rolling_sum.sql`  
Используется оконная функция `SUM() OVER (PARTITION BY ... ORDER BY ... ROWS BETWEEN ...)`.

Показываем первые 30 строк (полная таблица — по одной строке на каждый месяц каждого отправителя).

In [22]:
sql = (ROOT / "sql" / "rolling_sum.sql").read_text(encoding="utf-8")
df_rolling = run_query(sql, ROOT / "data" / "counterparty.db")
print(f"Всего строк в результате: {len(df_rolling):,}")
df_rolling.head(30)

Всего строк в результате: 49,667


,sen,mon,sum,rolling_sum
0,000103385227,2024-03,"55,257.61","55,257.61"
1,000103385227,2024-09,"144,124.44","199,382.05"
2,000103385227,2024-11,"23,653.16","223,035.21"
3,000103385227,2025-01,"10,440.91","233,476.12"
4,000103385227,2025-03,"2,745,540.03","2,979,016.15"
5,000103385227,2025-08,"68,892.16","3,047,908.31"
6,000103385227,2025-12,"4,008,535.20","7,056,443.51"
7,000103385227,2026-02,"618,046.75","7,674,490.26"
8,000106497273,2024-01,"6,640,227.80","6,640,227.80"
9,000106497273,2024-03,"2,740,385.48","9,380,613.28"


## 3. Контрагенты с концентрацией входящих >70%

Запрос: `sql/concentration.sql`  
Контрагенты, у которых более 70% всех входящих платежей приходит от одного источника — признак аномальной зависимости.

In [25]:
sql = (ROOT / "sql" / "concentration.sql").read_text(encoding="utf-8")
df_conc = run_query(sql, ROOT / "data" / "counterparty.db")
print(f"Контрагентов с концентрацией >70%: {len(df_conc)}")
df_conc

Контрагентов с концентрацией >70%: 221


,rec,главный_источник,sum,sum_total,доля_от_источника
0,771020553470,720903078056,"14,186,721.01","14,648,530.82",96.9%
1,260321585703,090504618028,"10,519,842.77","10,955,384.83",96.0%
2,040603497571,260321585703,"2,038,836.16","2,126,576.97",95.9%
3,361106421557,610308598441,"12,553,950.56","13,158,978.96",95.4%
4,441021449949,140512557513,"14,517,668.14","15,254,626.49",95.2%
...,...,...,...,...,...
216,780807445718,700205448349,"4,554,242.38","6,486,035.62",70.2%
217,141001534437,410712523791,"9,508,374.45","13,558,014.74",70.1%
218,590905044487,840502656986,"14,857,424.62","21,212,361.98",70.0%
219,571002501577,760919406851,"12,819,896.54","18,308,197.63",70.0%
